# Virtue Foundation — exploration

Local Parquet copies of the three Databricks tables. Re-download any time with
`python scripts/download_data.py`.

See `CLAUDE.md` for the challenge brief, data gotchas and the agreed plan.

In [9]:
import json
import pathlib

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 100)

DATA = pathlib.Path("..") / "data"

fac = pd.read_parquet(DATA / "facilities.parquet")
pin = pd.read_parquet(DATA / "india_post_pincode_directory.parquet")
nfhs = pd.read_parquet(DATA / "nfhs_5_district_health_indicators.parquet")

for name, df in [("facilities", fac), ("pincode", pin), ("nfhs5", nfhs)]:
    print(f"{name:12s} {df.shape[0]:>7,} rows x {df.shape[1]:>3} cols")

facilities    10,088 rows x  51 cols
pincode      165,627 rows x  11 cols
nfhs5            706 rows x 109 cols


## Parsing the JSON list columns

Every column in `facilities` is a string except `latitude` / `longitude`.
These six hold JSON-encoded arrays.

In [2]:
LIST_COLS = [
    "specialties",
    "procedure",
    "equipment",
    "capability",
    "phone_numbers",
    "source_urls",
]


def parse_list(value):
    """JSON string -> list of non-empty strings. Tolerates nulls and bad JSON."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    try:
        parsed = json.loads(value)
    except (TypeError, ValueError):
        return []
    if not isinstance(parsed, list):
        return []
    return [str(x).strip() for x in parsed if x and str(x).strip()]


for col in LIST_COLS:
    fac[col + "_list"] = fac[col].map(parse_list)

fac[[c + "_list" for c in LIST_COLS]].map(len).describe().T[["mean", "50%", "max"]]

,mean,50%,max
specialties_list,27.656027,24.0,50.0
procedure_list,16.236221,10.0,50.0
equipment_list,5.556305,2.0,50.0
capability_list,22.970956,18.0,50.0
phone_numbers_list,9.624603,6.0,50.0
source_urls_list,15.732752,8.0,50.0


## The evidence frame

One row per `(facility, snippet, source_field)` — ~460k rows.
This is the grain everything downstream should work at, because
**claims belong to sentences, not to records** (see next section).

In [10]:
fac

,unique_id,source_types,source_ids,source_content_id,name,organization_type,content_table_id,phone_numbers,officialPhone,email,websites,officialWebsite,yearEstablished,acceptsVolunteers,facebookLink,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,countries,facilityTypeId,operatorTypeId,affiliationTypeIds,description,area,numberDoctors,capacity,specialties,procedure,equipment,capability,recency_of_page_update,distinct_social_media_presence_count,affiliated_staff_presence,custom_logo_presence,number_of_facts_about_the_organization,post_metrics_most_recent_social_media_post_date,post_metrics_post_count,engagement_metrics_n_followers,engagement_metrics_n_likes,engagement_metrics_n_engagements,source,coordinates,latitude,longitude,cluster_id,source_urls
0,b8a5401f-42f1-422a-8cd9-686a15b4cb76,"[""dynamic"",""overture"",""constant"",""mongo_facility"",""dynamic"",""overture"",""constant"",""mongo_facility"",""dynamic"",""overture"",""constant"",""mongo_facility"",""dynamic...","[""08f42d99986e2b6d03862b16be6f4556"",""08f3cb3a2a1a139503b31af626a9e452"",""08f52b1ca500bb5c034e6665048ce729"",""605030acb13d98c445519412"",""08f6088501329d6203f835...",c2ada12b-11f7-45f2-93b6-94f7e8853c7a,Aravind Eye Hospital,facility,c2ada12b-11f7-45f2-93b6-94f7e8853c7a,"[""+911123233400"",""+911123232400"",""+911126142832"",""+911126141119"",""+911120895995"",""+919910631876"",""+911123236000"",""+911123236000"",""+911122890612"",""+911122890...",+911123233400,ighofficedwarka@gmail.com,"[""aravind.org"",""https://tgsouthernpower.org/creditcardlogin"",""https://health.delhi.gov.in/vacancy"",""https://hfw.assam.gov.in/frontimpotentdata/treatment-out...",aravind.org,1991,false,https://www.facebook.com/maharaja-agrasen-hospital-dwarka-sector-1-103371528653223,Santosh Nagar Colony,Delhi Gate,Delhi Gate,Hyderabad,Telangana,110002,India,IN,null,hospital,private,"[""government"",""government"",""government"",""government"",""government"",""government"",""government"",""government"",""government"",""government"",""government"",""government""...",Referral hospital outside the state listed under Assam Government schemes (AAN/Atal Amrit Abhiyan).,20000,1,650,"[""ophthalmology"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""inter...","[""Dialysis treatment is provided via a 20-bed Dialysis Unit with 10 machines as part of the Pradhan Mantri National Dialysis Programme"",""CT scan and MRI ima...","[""CT scanner"",""MRI scanner"",""Dialysis machines: 10"",""50 MT LMO storage tank"",""20 KL LMO storage tank"",""Gas pipeline system supplying all 1241 beds"",""Six PSA...","[""Eye care hospital"",""Located in Hyderabad"",""Aravind Eye Hospital Pvt. Ltd. is listed in the Hyderabad hospital directory as Aravind Eye Hospital"",""Madras M...",2027-07-20,6,true,true,73,2025-12-17,3,523,4,6,kie,"{""coordinates"":[79.48544311523438,11.93823528289795],""type"":""Point""}",11.938235,79.485443,000140e6-c506-4f6e-84dd-38b7b44bcbb0,"[""https://tgsouthernpower.org/creditcardlogin"",""https://health.delhi.gov.in/vacancy"",""https://hfw.assam.gov.in/frontimpotentdata/treatment-outside-assam"",""h..."
1,06d5fb63-b2f1-4001-9c7c-a9fea7241dc3,"[""overture"",""overture"",""dynamic"",""dynamic"",""dynamic"",""dynamic"",""dynamic"",""dynamic"",""dynamic"",""dynamic"",""overture"",""overture"",""dynamic"",""dynamic"",""overture"",...","[""08f3d8d52038c55403da8a47f904d2dc"",""08f3d8d52038c55403da8a47f904d2dc"",""08f611cb13466275038433e8fb7296f0"",""08f3da5570029a9903748f6b331f5ba7"",""08f426d9b30052...",a3168d54-bc65-4ebc-9553-91859010d1a7,"Fortis Hospital, Gurugram",facility,a3168d54-bc65-4ebc-9553-91859010d1a7,"[""+919289678787"",""+911247162200"",""+911244962200"",""+919582910304"",""+919693969347"",""+9118001031041"",""+919717041602"",""+919769516280"",""+919999722221"",""+91971704...",+919289678787,[email p

In [3]:
EVIDENCE_FIELDS = ["procedure", "capability", "equipment"]

evidence = pd.concat(
    [
        fac[["unique_id", "name", "address_city", "address_stateOrRegion", field + "_list"]]
        .rename(columns={field + "_list": "snippet"})
        .explode("snippet")
        .assign(source_field=field)
        for field in EVIDENCE_FIELDS
    ],
    ignore_index=True,
).dropna(subset=["snippet"])

print(f"{len(evidence):,} snippets")
evidence.source_field.value_counts()

451,574 snippets


source_field
capability    231731
procedure     163791
equipment      56052
Name: count, dtype: int64

## The core problem: snippets don't belong to their record

Records merge several hospitals. A naive `LIKE '%dialysis%'` on the record
returns facilities that do not offer dialysis at all.

In [5]:
hits = evidence[evidence.snippet.str.contains("dialysis", case=False, na=False)]

# Aravind Eye Hospital, Hyderabad -- an EYE hospital with a 20-bed dialysis unit?
hits[hits.name.str.contains("Aravind Eye", case=False, na=False)].head(3)[
    ["name", "address_city", "source_field", "snippet"]
]

,unique_id,name,address_city,address_stateOrRegion,snippet,source_field
0,b8a5401f-42f1-422a-8cd9-686a15b4cb76,Aravind Eye Hospital,Hyderabad,Telangana,Dialysis treatment is provided via a 20-bed Dialysis Unit with 10 machines as part of the Pradhan Mantri National Dialysis Programme,procedure
7,b8a5401f-42f1-422a-8cd9-686a15b4cb76,Aravind Eye Hospital,Hyderabad,Telangana,Hemodialysis is performed in three units as a regular service,procedure
8,b8a5401f-42f1-422a-8cd9-686a15b4cb76,Aravind Eye Hospital,Hyderabad,Telangana,Peritoneal dialysis is available around the clock,procedure


In [ ]:
# Snippets that name a DIFFERENT facility than the record they sit in.
# Crude proxy: the snippet says "at <X> Hospital" and <X> is not in the record name.
named = evidence.snippet.str.extract(
    r"\bat ([A-Z][A-Za-z.'&\- ]{3,40}(?:Hospital|Institute|Centre|Center|Clinic))"
)[0]

mismatch = evidence.assign(named_facility=named).dropna(subset=["named_facility"])
mismatch = mismatch[
    ~mismatch.apply(
        lambda r: r.named_facility.split()[0].lower() in str(r["name"]).lower(), axis=1
    )
]

print(f"{len(mismatch):,} snippets name a facility that doesn't match their record")
print(f"across {mismatch.unique_id.nunique():,} of {fac.unique_id.nunique():,} facilities")
mismatch[["name", "named_facility", "snippet"]].head(8)

## Specialties — the inherited taxonomy

Already a clean controlled vocabulary. Note these are **departments**, not services:
`nephrology` is not `dialysis`.

In [ ]:
spec = (
    fac[["unique_id", "specialties_list"]]
    .explode("specialties_list")
    .dropna()
    .drop_duplicates()  # values repeat within a row
    .rename(columns={"specialties_list": "specialty"})
)

print(f"{spec.specialty.nunique()} distinct specialties")
spec.specialty.value_counts().head(25)

## Geography

Coordinates are trustworthy (99% present). Place names are not —
`address_stateOrRegion` has far more distinct values than India has states.

In [ ]:
print(f"geocoded:            {fac.latitude.notna().sum():,} / {len(fac):,}")
print(f"distinct 'states':   {fac.address_stateOrRegion.nunique():,}  (India has 37)")
print(f"distinct cities:     {fac.address_city.nunique():,}")

# PIN -> coordinates lookup, deduplicated (source grain is post office, not PIN)
pin_geo = (
    pin.assign(
        lat=pd.to_numeric(pin.latitude, errors="coerce"),
        lon=pd.to_numeric(pin.longitude, errors="coerce"),
    )
    .dropna(subset=["lat", "lon"])
    .groupby("pincode")
    .agg(lat=("lat", "median"), lon=("lon", "median"), district=("district", "first"),
         state=("statename", "first"))
)
print(f"\nPIN lookup rows:     {len(pin_geo):,}")
pin_geo.head()